# EnvPool

> Steps several wrapped multi-agent envs together and stacks their info dicts.

In [ ]:
#| default_exp world.env_pool

In [ ]:
#| hide
from nbdev.showdoc import *

The multi-agent counterpart of stable-worldmodel's `EnvPool`. The envs must be wrapped with
`MegaWrapper` (as `World` does), so each returns one flat info dict
(see `wrappers/02_default.ipynb`). The pool stacks these into one dict whose values have a
leading ``(num_envs, 1)`` (env, time): per-agent values are ``(num_envs, 1, num_agents, ...)``.

The stacked buffers are allocated on the first reset and written in place afterwards (copy what
you keep).

In [ ]:
#| export
from __future__ import annotations

from typing import Any, Callable, Sequence

import gymnasium as gym
import numpy as np
from gymnasium import spaces
from gymnasium.vector.utils import batch_space

In [ ]:
#| export
def joint_action_space(env: gym.Env) -> spaces.Space:
    """
    Action space of all the agents of `env` as one space: MultiDiscrete ``(num_agents,)`` if
    they are all Discrete, a Box ``(num_agents, *shape)`` if they are all Boxes of one shape.
    """
    agent_spaces = [env.action_space[a] for a in range(env.unwrapped.num_agents)]
    if all(isinstance(s, spaces.Discrete) for s in agent_spaces):
        return spaces.MultiDiscrete([s.n for s in agent_spaces], start=[s.start for s in agent_spaces])
    if all(isinstance(s, spaces.Box) for s in agent_spaces) and len({s.shape for s in agent_spaces}) == 1:
        return spaces.Box(low=np.stack([s.low for s in agent_spaces]), high=np.stack([s.high for s in agent_spaces]),
                          dtype=agent_spaces[0].dtype)
    return spaces.Tuple(agent_spaces)


class EnvPool:
    """
    Runs ``num_envs`` wrapped multi-agent envs side by side, one after the other in this process.

    :meth:`reset` and :meth:`step` take a `mask` to only reset / step some envs: the others keep
    their info and report zero reward and no termination.

    Parameters
    ----------
    env_fns : list of callables
        One factory per env, each called once; the envs must return flat info dicts
        (wrapped with :class:`MegaWrapper`)
    """
    def __init__(self, env_fns: Sequence[Callable[[], gym.Env]]):
        self.envs: list[gym.Env] = [fn() for fn in env_fns]
        assert self.envs, "EnvPool needs at least one env"
        self.num_agents: int = self.envs[0].unwrapped.num_agents
        assert all(env.unwrapped.num_agents == self.num_agents for env in self.envs), \
            "All envs of a pool must have the same number of agents"
        self.seeds = np.full(self.num_envs, -1, dtype=np.int64)
        self._single_action_space = joint_action_space(self.envs[0])
        # cached: a batched space rebuilt on each access would never advance its RNG
        self._action_space = batch_space(self._single_action_space, self.num_envs)
        self._stacked_infos: dict[str, Any] | None = None

    @property
    def num_envs(self) -> int:
        return len(self.envs)

    @property
    def action_space(self) -> spaces.Space:
        "Batched action space: samples have shape (num_envs, num_agents)."
        return self._action_space

    @property
    def single_action_space(self) -> spaces.Space:
        "Joint action space of the agents of one env."
        return self._single_action_space

    @property
    def single_observation_space(self) -> spaces.Dict:
        "Observation space of one env (agent id -> agent observation space)."
        return self.envs[0].observation_space

    @property
    def state_space(self) -> spaces.Space:
        "Global state space of one env."
        return self.envs[0].unwrapped.state_space

    @property
    def variation_space(self):
        "Variation space of the envs, or None if they do not define one."
        return getattr(self.envs[0].unwrapped, 'variation_space', None)

    def close(self):
        for env in self.envs:
            env.close()

In [ ]:
#| export
def _broadcast_arg(arg, n: int, increment: bool = False) -> list:
    "One value per env: lists are used as is, an int seed becomes `seed + i`, anything else is shared."
    if isinstance(arg, (list, tuple, np.ndarray)):
        assert len(arg) == n, f"Expected {n} values, got {len(arg)}"
        return list(arg)
    if increment and isinstance(arg, (int, np.integer)):
        return [int(arg) + i for i in range(n)]
    return [arg] * n


def _check_info(info: Any):
    if not isinstance(info, dict) or 'step_idx' not in info:
        raise TypeError("EnvPool envs must return flat info dicts: wrap them with MegaWrapper "
                        "(stable_marl.wrappers.default), as World does")


def _stack_fresh(per_env_infos: list[dict]) -> dict[str, Any]:
    "Stacked info buffers, (num_envs, 1, ...) for arrays and numbers, nested lists otherwise."
    stacked = {}
    for key in per_env_infos[0]:
        values = [info[key] for info in per_env_infos]
        if isinstance(values[0], (np.ndarray, np.generic, bool, int, float)) and not isinstance(values[0], str):
            stacked[key] = np.stack([np.asarray(v) for v in values])[:, None]
        else:
            stacked[key] = [[v] for v in values]
    return stacked


def _write_env_info(stacked: dict, i: int, info: dict):
    "Write the info of env `i` into the stacked buffers, in place."
    for key, value in info.items():
        if key in stacked:
            stacked[key][i][0] = value


def episode_done(terminateds: np.ndarray, truncateds: np.ndarray) -> np.ndarray:
    "Whether each env's episode is over (every agent terminated or truncated), shape (num_envs,)."
    return (np.asarray(terminateds) | np.asarray(truncateds)).all(axis=1)

In [ ]:
#| export
from fastcore.utils import patch

@patch
def reset(
    self: EnvPool,
    seed: int | Sequence[int | None] | None = None,
    options: dict | Sequence[dict | None] | None = None,
    mask: np.ndarray | None = None) -> tuple[None, dict]:
    """
    Reset the envs (only those where `mask` is True, if given) and return ``(None, infos)``.

    `seed` is an int (env `i` gets ``seed + i``), one seed per env, or None.
    `options` is one dict for every env, or one per env.
    """
    seeds, options = _broadcast_arg(seed, self.num_envs, increment=True), _broadcast_arg(options, self.num_envs)
    per_env_infos = [None] * self.num_envs
    for i in (range(self.num_envs) if mask is None else np.flatnonzero(mask)):
        _, per_env_infos[i] = self.envs[i].reset(seed=seeds[i], options=options[i])
        _check_info(per_env_infos[i])
        self.seeds[i] = -1 if seeds[i] is None else seeds[i]
    if self._stacked_infos is None or mask is None:
        if any(info is None for info in per_env_infos):
            raise RuntimeError("The first reset of an EnvPool must reset every env")
        self._stacked_infos = _stack_fresh(per_env_infos)
    else:
        for i, info in enumerate(per_env_infos):
            if info is not None:
                _write_env_info(self._stacked_infos, i, info)
    return None, self._stacked_infos

@patch
def step(
    self: EnvPool,
    actions: np.ndarray,
    mask: np.ndarray | None = None) -> tuple[None, np.ndarray, np.ndarray, np.ndarray, dict]:
    """
    Step the envs (only those where `mask` is True, if given) with `actions` of shape
    ``(num_envs, num_agents)``. Returns ``(None, rewards, terminateds, truncateds, infos)``,
    the first three of shape ``(num_envs, num_agents)``.
    """
    actions = np.asarray(actions)
    assert actions.shape[:2] == (self.num_envs, self.num_agents), \
        f"Expected actions of shape ({self.num_envs}, {self.num_agents}), got {actions.shape}"
    shape = (self.num_envs, self.num_agents)
    rewards, terminateds, truncateds = np.zeros(shape, np.float32), np.zeros(shape, bool), np.zeros(shape, bool)
    for i in (range(self.num_envs) if mask is None else np.flatnonzero(mask)):
        act = {a: actions[i, a].item() if actions[i, a].ndim == 0 else actions[i, a] for a in range(self.num_agents)}
        _, rew, term, trunc, info = self.envs[i].step(act)
        _write_env_info(self._stacked_infos, i, info)
        for a in range(self.num_agents):
            rewards[i, a], terminateds[i, a], truncateds[i, a] = rew[a], term[a], trunc[a]
    return None, rewards, terminateds, truncateds, self._stacked_infos

### Tests

In [ ]:
from fastcore.test import test_fail
from stable_marl.envs import make
from stable_marl.wrappers.default import MegaWrapper

wrapped = lambda: MegaWrapper(make('MultiGrid-Empty-6x6-v0', agents=2, render_mode='rgb_array'), add_pixels=True)
pool = EnvPool([wrapped] * 3)
_, infos = pool.reset(seed=10)
assert list(pool.seeds) == [10, 11, 12] and list(infos['seed'][:, 0]) == [10, 11, 12]
assert infos['image'].shape == (3, 1, 2, 7, 7, 3) and infos['pov'].shape == (3, 1, 2, 224, 224, 3)
assert infos['state'].shape == (3, 1, 6, 6, 3) and infos['pixels'].shape == (3, 1, 192, 192, 3)
assert np.isnan(infos['reward']).all() and np.isnan(infos['action']).all() and not infos['terminated'].any()
assert pool.action_space.sample().shape == (3, 2) and pool.single_action_space.shape == (2,)

# the pool matches stepping the envs one by one
solo = [make('MultiGrid-Empty-6x6-v0', agents=2) for _ in range(3)]
for i, env in enumerate(solo): env.reset(seed=10 + i)
rng = np.random.default_rng(0)
for t in range(20):
    actions = rng.integers(3, size=(3, 2))
    _, rew, term, trunc, infos = pool.step(actions)
    for i, env in enumerate(solo):
        o, r, te, tr, _ = env.step({a: int(actions[i, a]) for a in range(2)})
        for a in range(2):
            assert np.array_equal(infos['pov'][i, 0, a], o[a]['pov']) and rew[i, a] == r[a] and term[i, a] == te[a]
        assert np.array_equal(infos['state'][i, 0], env.state()) and infos['step_idx'][i, 0] == t + 1
    assert np.array_equal(infos['action'][:, 0], actions) and np.array_equal(infos['reward'][:, 0], rew)

# masked envs are not stepped
before = infos['image'].copy()
_, rew, term, trunc, infos = pool.step(np.full((3, 2), 2), mask=np.array([True, False, False]))
assert np.array_equal(infos['image'][1:], before[1:]) and not rew[1:].any() and not term[1:].any()

test_fail(lambda: EnvPool([lambda: make('MultiGrid-Empty-6x6-v0')]).reset(), contains='MegaWrapper')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()